In [94]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

In [95]:
def print_shapes(*args):
    for a in args:
        print("->",a.shape,"|", a.dtype, end="\n")

In [96]:
digits_img = cv.imread("../../../class.vision/dataset/digits.png",0)
h, w = digits_img.shape[:2]
digits = digits_img.reshape(50, 20, 100, 20).transpose(0, 2, 1, 3).reshape(-1, 20, 20)
labels = np.repeat(np.arange(10), 500)
print(digits.shape)
print(labels.shape)

rnd = np.random.default_rng(64)
indices = rnd.permutation(len(digits))
digits = digits[indices]
labels =  labels[indices]

(5000, 20, 20)
(5000,)


In [97]:
winSize = (20,20)
blockSize = (8,8)
blockStride = (4,4)
cellSize = (8,8)
nbins = 9
derivAperture = 1
winSigma = -1.
histogramNormType = 0
L2HysThreshold = 0.2
gammaCorrection = 1
nlevels = 64
signedGradient = True

hog = cv.HOGDescriptor(winSize,blockSize,blockStride,cellSize,nbins,derivAperture,winSigma,histogramNormType,L2HysThreshold,gammaCorrection,nlevels, signedGradient)

In [98]:
descriptors = []
for digit in digits:
    descriptor = hog.compute(digit)
    descriptors.append(descriptor)
descriptors = np.array(descriptors)
print(descriptors.shape)

(5000, 144)


In [99]:
x_train, x_test = np.vsplit(descriptors, 2)
y_train, y_test = np.vsplit(labels[:, np.newaxis], 2)
x_train, x_test = x_train.astype(np.float32), x_test.astype(np.float32)
print_shapes(x_train, y_train, x_test, y_test)

-> (2500, 144) | float32
-> (2500, 1) | int64
-> (2500, 144) | float32
-> (2500, 1) | int64


In [100]:
C=12.5
gamma=0.50625
model = cv.ml.SVM_create()
model.setGamma(gamma)
model.setC(C)
model.setKernel(cv.ml.SVM_RBF)
model.setType(cv.ml.SVM_C_SVC)

In [101]:
model.train(x_train, cv.ml.ROW_SAMPLE, y_train)

True

In [102]:
predictions = model.predict(x_test)[1]
accuracy = (y_test == predictions).mean()
print('Percentage Accuracy: %.2f %%' % (accuracy*100))

Percentage Accuracy: 98.20 %
